<a href="https://colab.research.google.com/github/shahzaib-Ali59/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/shahzaib-Ali59/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("Working dir:", os.getcwd())
print(df.shape[0], "pages loaded")

Working dir: /content/flyrank-ml-internship/flyrank-ml-internship
30000 pages loaded


# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahzaib-Ali59/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Type: Scoring / Ranking**

My lane (Refresh / Content Opportunity Scoring) is fundamentally a scoring problem: every
page gets a score representing how urgently it needs review, and pages are then ranked by
that score. Under the hood, I use a classification model (predicting whether a page is
declining) to produce that score -- but the actual deliverable is a *ranked list*, not a
single yes/no label. A content reviewer with limited time cares about "which pages first,"
not "which pages are declining" in isolation.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

**Target: `is_declining_label`**, defined as `1` when `trend_direction == "down"`, else `0`.

This is a **proxy**, not a directly observed "needs review" outcome -- no one has manually
labeled which pages truly need attention. `trend_direction` is itself derived from `trend_pct`
in the dataset, so I'm using an observed trend signal as a stand-in for the real thing I care
about (business-relevant decline). This is a reasonable proxy but not a perfect one -- a page
could show "down" trend from normal seasonality rather than genuine decay.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

**Precision@50.**

A content reviewer can only realistically look at a fixed number of pages per cycle -- I'm
assuming around 50. So what matters isn't overall accuracy across all 30,000 pages, it's:
of the top 50 pages my model flags, how many are actually declining? In Week 1, the hand rule
scored Precision@50 = 0.240, the random forest scored 0.740. That is a defensible,
decision-relevant number because it directly measures the quality of the reviewer's actual
workflow -- not some abstract score no one acts on.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

One row = one page (`content_id` is the unique identifier). Loading the lane's relevant
slice of columns below.

In [ ]:
lane_cols = ["content_id", "trend_direction", "is_declining_label",
             "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]

lane_df = df[lane_cols].copy()
print("Unit of analysis: one row = one page")
print(f"Shape: {lane_df.shape[0]} rows, {lane_df.shape[1]} columns")
lane_df.head(5)

Unit of analysis: one row = one page
Shape: 30000 rows, 8 columns


,content_id,trend_direction,is_declining_label,days_since_last_update,impressions_90d,avg_position,ctr,word_count
0,content_304f48230142,down,1,20,3803,10.6,0.76,3221.0
1,content_a1fb4e703a9e,down,1,25,15320,20.3,0.05,2481.0
2,content_9aa793d4d895,down,1,20,12581,36.5,0.09,3515.0
3,content_331d6c4de07b,stable,0,22,11751,6.2,0.49,NaN
4,content_d99b7a2d90ca,down,1,14,19140,44.0,0.13,2803.0


## 5. Why ML beats a fixed rule here

A fixed rule (like "flag pages not updated in 180+ days with high impressions") only
considers a couple of variables in isolation, with a hand-picked threshold. Real decline is
driven by an *interaction* of many signals at once -- position, CTR, content age, update
recency -- and the right threshold for one signal often depends on the value of another.
That's exactly the kind of pattern a fixed if-statement can't capture, but a model that
learns from data can. The numbers below make this concrete again.

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
hand_rule_score = stale * visible * df["impressions_90d"]

y = df["is_declining_label"].values
hand_rule_p50 = precision_at_k(hand_rule_score, y, 50)

print(f"Hand-written rule Precision@50: {hand_rule_p50:.3f}")
print("From Week 1: a random forest scored Precision@50 = 0.740 on the same task")
print(f"\nGap: the learned model is roughly {0.740/hand_rule_p50:.1f}x better than the fixed rule.")

Hand-written rule Precision@50: 0.680
From Week 1: a random forest scored Precision@50 = 0.740 on the same task

Gap: the learned model is roughly 1.1x better than the fixed rule.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.